# LeetCode 2881 — Create a New Column bằng Python (pandas)

Link đề: https://leetcode.com/problems/create-a-new-column/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thêm cột mới `bonus` bằng **gấp đôi** cột `salary` rồi trả về DataFrame.
2. Biết **gán cột** (`df["cột_mới"] = ...`) và hiểu **phép toán theo từng phần tử trên cả cột** (vector hóa): `df["salary"] * 2`.
3. Đi từ cách **vòng lặp thuần Python** → **vector hóa `df["salary"] * 2`** → **`assign` (không sửa bảng gốc)** → **`apply` / `map` từng phần tử** → **`insert` và `eval`**.
4. Phân biệt **sửa tại chỗ** (in-place, thay đổi bảng người gọi) với **trả về bản sao** (`assign`), và biết khi nào quan trọng.
5. Tránh các lỗi kinh điển: dùng `df.bonus = ...` (không tạo cột), quên gán, quên dấu nháy quanh tên cột, nhân `2.0` làm đổi dtype, độ dài list không khớp.

## 2. Đề bài & ví dụ

Một công ty muốn thưởng cho nhân viên: tiền thưởng (`bonus`) bằng **gấp đôi lương** (`salary`).

Cho DataFrame `employees` có các cột `name` và `salary`. Hãy tạo **một cột mới** tên `bonus` chứa **gấp đôi giá trị `salary`** của mỗi nhân viên rồi trả về DataFrame.

**Ví dụ**

- Input: DataFrame `employees`

| name | salary |
|---|---|
| Piper | 4548 |
| Grace | 28150 |
| Georgia | 1103 |
| Willow | 6593 |
| Finn | 74576 |
| Thomas | 24433 |

- Output:

| name | salary | bonus |
|---|---|---|
| Piper | 4548 | 9096 |
| Grace | 28150 | 56300 |
| Georgia | 1103 | 2206 |
| Willow | 6593 | 13186 |
| Finn | 74576 | 149152 |
| Thomas | 24433 | 48866 |

- Giải thích: cột `bonus` có giá trị bằng 2 lần `salary` ở mỗi hàng.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    "name": ["Piper", "Grace", "Georgia", "Willow", "Finn", "Thomas"],
    "salary": [4548, 28150, 1103, 6593, 74576, 24433],
})
print(employees)
print(employees.dtypes)

      name  salary
0    Piper    4548
1    Grace   28150
2  Georgia    1103
3   Willow    6593
4     Finn   74576
5   Thomas   24433
name        str
salary    int64
dtype: object


## 3. Kiến thức nền

### 3.1. Cột là một `Series`

Mỗi cột của DataFrame là một **Series**: một dãy giá trị có kèm nhãn (index). `df["salary"]` lấy cột đó ra.

In [2]:
s = employees["salary"]
print(type(s).__name__)
print(s)
print(s.tolist())

Series
0     4548
1    28150
2     1103
3     6593
4    74576
5    24433
Name: salary, dtype: int64
[4548, 28150, 1103, 6593, 74576, 24433]


### 3.2. Phép toán trên **cả cột** (vector hóa / broadcasting)

Khi viết `s * 2`, pandas nhân **từng phần tử** của Series với `2` và trả về Series mới cùng độ dài. Không cần vòng lặp `for`. Ta gọi đây là **vector hóa**, nhanh vì chạy bằng mã C.

In [3]:
print(s * 2)
print((s + 100).tolist())
print((s / 1000).tolist())          # chia thường -> số thực
print((s // 1000).tolist())         # chia lấy phần nguyên -> số nguyên

0      9096
1     56300
2      2206
3     13186
4    149152
5     48866
Name: salary, dtype: int64
[4648, 28250, 1203, 6693, 74676, 24533]
[4.548, 28.15, 1.103, 6.593, 74.576, 24.433]
[4, 28, 1, 6, 74, 24]


### 3.3. Gán cột mới: `df["tên"] = giá_trị`

Gán vào một **khóa chưa tồn tại** sẽ **tạo cột mới ở cuối bảng**; gán vào khóa đã tồn tại sẽ **ghi đè** cột đó. Giá trị gán có thể là Series (căn theo index), list (cùng độ dài) hay một số (điền cho mọi hàng).

In [4]:
df = employees.copy()
df["bonus"] = df["salary"] * 2          # từ Series
df["level"] = "A"                        # một giá trị cho mọi hàng
df["rank"] = [1, 2, 3, 4, 5, 6]          # list cùng độ dài
print(df)
print(df.columns.tolist())

      name  salary   bonus level  rank
0    Piper    4548    9096     A     1
1    Grace   28150   56300     A     2
2  Georgia    1103    2206     A     3
3   Willow    6593   13186     A     4
4     Finn   74576  149152     A     5
5   Thomas   24433   48866     A     6
['name', 'salary', 'bonus', 'level', 'rank']


### 3.4. Sửa tại chỗ hay trả bản sao: `df[...] = ...` và `df.assign(...)`

- `df["bonus"] = ...` **sửa trực tiếp** DataFrame đang cầm (người gọi hàm cũng thấy cột mới).
- `df.assign(bonus=...)` **trả về bảng mới** có thêm cột; bảng gốc **không** đổi. Tên cột là **tên tham số**; giá trị có thể là Series, số, hoặc hàm `lambda df: ...`.

In [5]:
a = employees.copy()
b = a.assign(bonus=a["salary"] * 2)
print(a.columns.tolist(), "| gốc không đổi")
print(b.columns.tolist(), "| bảng mới có bonus")

a["bonus"] = a["salary"] * 2
print(a.columns.tolist(), "| sau khi gán trực tiếp, a đã đổi")

['name', 'salary'] | gốc không đổi
['name', 'salary', 'bonus'] | bảng mới có bonus
['name', 'salary', 'bonus'] | sau khi gán trực tiếp, a đã đổi


### 3.5. `apply` / `map`: áp dụng hàm cho từng phần tử

`s.apply(f)` (hoặc `s.map(f)`) gọi hàm Python `f` cho **từng** phần tử. Linh hoạt nhưng **chậm** hơn vector hóa vì mỗi phần tử gọi một hàm Python.

In [6]:
print(employees["salary"].apply(lambda x: x * 2).tolist())
print(employees["salary"].map(lambda x: x * 2).tolist())

[9096, 56300, 2206, 13186, 149152, 48866]
[9096, 56300, 2206, 13186, 149152, 48866]


### 3.6. `df.insert(vị_trí, tên, giá_trị)` và `df.eval`

- `df.insert(k, "tên", giá_trị)` chèn cột **tại vị trí `k`** (sửa tại chỗ; báo lỗi nếu tên cột đã có).
- `df.eval("bonus = salary * 2")` đánh giá biểu thức chuỗi và **trả về bảng mới** có cột `bonus`.

In [7]:
x = employees.copy()
x.insert(1, "bonus", x["salary"] * 2)            # chèn vào vị trí 1 (giữa name và salary)
print(x.columns.tolist())
print(employees.eval("bonus = salary * 2").columns.tolist())

['name', 'bonus', 'salary']
['name', 'salary', 'bonus']


### 3.7. Kiểu dữ liệu (dtype) khi tính toán

Nhân số nguyên với số nguyên (`* 2`) giữ **`int64`**. Nhân với số thực (`* 2.0`) hoặc chia thường (`/`) đổi sang **`float64`**. Bộ chấm so cả dtype, nên `bonus` phải là **số nguyên**.

In [8]:
print((employees["salary"] * 2).dtype)          # int64
print((employees["salary"] * 2.0).dtype)        # float64
print((employees["salary"] / 1).dtype)          # float64

int64
float64
float64


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với mỗi hàng, `bonus = 2 × salary`:

| name | salary | bonus = 2 × salary |
|---|---|---|
| Piper | 4548 | 9096 |
| Grace | 28150 | 56300 |
| Georgia | 1103 | 2206 |

Ba điểm cần nhớ:

1. Công thức áp dụng **độc lập cho từng hàng**; không hàng nào ảnh hưởng hàng khác.
2. Cột mới nằm **bên phải** các cột cũ: `name | salary | bonus`.
3. Các cột cũ (`name`, `salary`) **không đổi**; chỉ thêm một cột.

Vì mọi hàng dùng cùng công thức, pandas cho phép viết một lần: "cột `bonus` = cột `salary` × 2".

## 5. Cách 1 — Brute Force / baseline: vòng lặp Python từng hàng

Duyệt từng lương, tính gấp đôi, gom vào list `bonus`, rồi gán list đó thành cột mới (cùng độ dài với bảng).

In [9]:
def bonus_loop(employees: pd.DataFrame) -> pd.DataFrame:
    bonus = []
    for salary in employees["salary"]:
        bonus.append(salary * 2)
    employees["bonus"] = bonus            # gán list cùng độ dài làm cột mới
    return employees

demo = bonus_loop(employees.copy())
print(demo)
print(demo.dtypes.tolist())

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866
[<StringDtype(storage='python', na_value=nan)>, dtype('int64'), dtype('int64')]


**Độ phức tạp:** duyệt `m` hàng, mỗi hàng một phép nhân và một `append`: **Time `O(m)`** (nhưng chạy bằng **Python thuần**, chậm), **Space `O(m)`** cho list và cột mới.

**Nút thắt:** vòng lặp Python lặp lại việc mà pandas có thể làm cho cả cột trong một lệnh bằng mã C (nhanh hơn hàng chục lần với bảng lớn).

## 6. Ý tưởng thuật toán / bước đột phá

**Vector hóa:** `employees["salary"] * 2` nhân cả cột trong một lần, ra Series mới cùng index. Gán Series đó vào `employees["bonus"]` — pandas **căn theo index** nên mỗi hàng nhận đúng giá trị của mình.

```
employees["bonus"] = employees["salary"] * 2
```

Có ba "phong cách" đạt cùng kết quả:

- **Sửa tại chỗ:** `df["bonus"] = ...` (Cách 2).
- **Trả về bảng mới, không đụng bảng gốc:** `df.assign(bonus=...)` (Cách 3).
- **Hàm cho từng phần tử:** `apply` / `map` (Cách 4) — dùng khi công thức phức tạp hơn nhân với 2.

## 7. Các cách tiếp theo

### Cách 2 — Vector hóa và gán cột (cách chuẩn)

**Giải thích từng đoạn.**

- `employees["salary"] * 2`: Series gấp đôi lương.
- `employees["bonus"] = ...`: tạo cột `bonus` ở cuối bảng.
- `return employees`: trả về bảng (đã được sửa tại chỗ).

In [10]:
def bonus_vectorized(employees: pd.DataFrame) -> pd.DataFrame:
    employees["bonus"] = employees["salary"] * 2
    return employees

demo = bonus_vectorized(employees.copy())
print(demo)

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


In [11]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["bonus"] = employees["salary"] * 2
    return employees

print(createBonusColumn(employees.copy()))

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


**Độ phức tạp Cách 2:** **Time `O(m)`** (vector hóa, nhanh), **Space `O(m)`** cho cột `bonus`.

---

### Cách 3 — `assign` (trả về bảng mới, không sửa bảng gốc)

**Giải thích.** `employees.assign(bonus=employees["salary"] * 2)` tạo bản mới có thêm cột `bonus`. Bảng người gọi **giữ nguyên**: hợp lối viết "hàm thuần" (không tác dụng phụ).

In [12]:
def bonus_assign(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.assign(bonus=employees["salary"] * 2)

src = employees.copy()
out = bonus_assign(src)
print(out.columns.tolist(), "| bảng nguồn:", src.columns.tolist())

['name', 'salary', 'bonus'] | bảng nguồn: ['name', 'salary']


In [13]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.assign(bonus=employees["salary"] * 2)

print(createBonusColumn(employees))

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


**Độ phức tạp Cách 3:** **Time `O(m·c)`** (sao chép các cột), **Space `O(m·c)`** — pandas 3.x (Copy-on-Write) sao chép "lười" nên thực tế nhẹ hơn. Dùng khi cần **không sửa** dữ liệu đầu vào.

---

### Cách 4 — `apply` / `map` từng phần tử

**Giải thích.** Mỗi phần tử `x` của cột `salary` được truyền vào hàm `lambda x: x * 2`. Hữu ích khi công thức cần `if/else` hoặc hàm Python phức tạp; nhưng với phép nhân đơn giản thì Cách 2 tốt hơn.

In [14]:
def bonus_apply(employees: pd.DataFrame) -> pd.DataFrame:
    employees["bonus"] = employees["salary"].apply(lambda x: x * 2)
    return employees

print(bonus_apply(employees.copy()))

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


In [15]:
# Code nộp LeetCode cho Cách 4
import pandas as pd

def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["bonus"] = employees["salary"].map(lambda x: x * 2)
    return employees

print(createBonusColumn(employees.copy()))

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


**Độ phức tạp Cách 4:** **Time `O(m)`** nhưng mỗi phần tử gọi một hàm Python nên **chậm hơn** Cách 2; **Space `O(m)`**.

---

### Cách 5 — `insert` (chọn vị trí cột) và `eval`

**Giải thích.**

- `employees.insert(len(employees.columns), "bonus", ...)` chèn cột **ở cuối** (hoặc đổi vị trí tùy ý). Tiện khi cột mới phải đứng ở một vị trí cụ thể.
- `employees.eval("bonus = salary * 2")` cho bảng mới, công thức viết bằng chuỗi.

In [16]:
def bonus_insert(employees: pd.DataFrame) -> pd.DataFrame:
    employees.insert(len(employees.columns), "bonus", employees["salary"] * 2)
    return employees

def bonus_eval(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.eval("bonus = salary * 2")

print(bonus_insert(employees.copy()).columns.tolist())
print(bonus_eval(employees).columns.tolist())

['name', 'salary', 'bonus']
['name', 'salary', 'bonus']


In [17]:
# Code nộp LeetCode cho Cách 5
import pandas as pd

def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees.insert(len(employees.columns), "bonus", employees["salary"] * 2)
    return employees

print(createBonusColumn(employees.copy()))

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


**Độ phức tạp Cách 5:** **Time `O(m)`**, **Space `O(m)`**. Chú ý: `insert` ném `ValueError` nếu cột `bonus` **đã tồn tại** (khác với gán trực tiếp là ghi đè).

## 8. TRACE từng bước

In DataFrame **sau mỗi bước** của Cách 2: bảng ban đầu → cột `salary` → Series `salary * 2` → sau khi gán cột.

In [18]:
def trace_vectorized(employees):
    df = employees.copy()
    print("BƯỚC 0 - bảng ban đầu:", df.shape, df.columns.tolist())
    print(df)

    salary = df["salary"]
    print("\nBƯỚC 1 - df['salary'] (một Series):")
    print(salary.tolist())

    doubled = salary * 2
    print("\nBƯỚC 2 - df['salary'] * 2 (Series mới, chưa nằm trong bảng):")
    print(doubled.tolist())
    print("cột bonus có trong bảng chưa?", "bonus" in df.columns)

    df["bonus"] = doubled
    print("\nBƯỚC 3 - df['bonus'] = ... (cột mới ở cuối):", df.shape, df.columns.tolist())
    print(df)
    print("dtypes:", df.dtypes.astype(str).tolist())
    return df

_ = trace_vectorized(employees)

BƯỚC 0 - bảng ban đầu: (6, 2) ['name', 'salary']
      name  salary
0    Piper    4548
1    Grace   28150
2  Georgia    1103
3   Willow    6593
4     Finn   74576
5   Thomas   24433

BƯỚC 1 - df['salary'] (một Series):
[4548, 28150, 1103, 6593, 74576, 24433]

BƯỚC 2 - df['salary'] * 2 (Series mới, chưa nằm trong bảng):
[9096, 56300, 2206, 13186, 149152, 48866]
cột bonus có trong bảng chưa? False

BƯỚC 3 - df['bonus'] = ... (cột mới ở cuối): (6, 3) ['name', 'salary', 'bonus']
      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866
dtypes: ['str', 'int64', 'int64']


In [19]:
def trace_loop(employees):
    bonus = []
    for i, (name, salary) in enumerate(zip(employees["name"], employees["salary"])):
        bonus.append(salary * 2)
        print(f"hàng {i}: {name:<8} salary={salary:>6} -> bonus={salary * 2:>7}   (list bonus hiện có {len(bonus)} phần tử)")
    out = employees.copy()
    out["bonus"] = bonus
    print(out)
    return out

_ = trace_loop(employees)

hàng 0: Piper    salary=  4548 -> bonus=   9096   (list bonus hiện có 1 phần tử)
hàng 1: Grace    salary= 28150 -> bonus=  56300   (list bonus hiện có 2 phần tử)
hàng 2: Georgia  salary=  1103 -> bonus=   2206   (list bonus hiện có 3 phần tử)
hàng 3: Willow   salary=  6593 -> bonus=  13186   (list bonus hiện có 4 phần tử)
hàng 4: Finn     salary= 74576 -> bonus= 149152   (list bonus hiện có 5 phần tử)
hàng 5: Thomas   salary= 24433 -> bonus=  48866   (list bonus hiện có 6 phần tử)
      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


Bảng tóm tắt trạng thái:

| bước | biểu thức | kết quả |
|---|---|---|
| 0 | `employees` | 6 hàng, cột `name`, `salary` |
| 1 | `employees["salary"]` | Series 6 số |
| 2 | `* 2` | Series 6 số (gấp đôi), chưa vào bảng |
| 3 | `employees["bonus"] = ...` | 6 hàng, 3 cột; `bonus` đứng cuối |

## 9. Code nộp LeetCode

In [20]:
import pandas as pd

def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["bonus"] = employees["salary"] * 2
    return employees

result = createBonusColumn(employees.copy())
print(result)
result

      name  salary   bonus
0    Piper    4548    9096
1    Grace   28150   56300
2  Georgia    1103    2206
3   Willow    6593   13186
4     Finn   74576  149152
5   Thomas   24433   48866


,name,salary,bonus
0,Piper,4548,9096
1,Grace,28150,56300
2,Georgia,1103,2206
3,Willow,6593,13186
4,Finn,74576,149152
5,Thomas,24433,48866


## 10. Vì sao thuật toán đúng

- **Đúng giá trị:** với mỗi hàng, `employees["salary"] * 2` cho `2 × salary` của đúng hàng đó; gán vào `bonus` **căn theo index**, nên không bị lệch hàng.
- **Đúng cột / vị trí:** gán vào khóa mới `"bonus"` tạo cột cuối cùng; các cột cũ giữ nguyên thứ tự và dữ liệu.
- **Kiểu dữ liệu:** `int64 * int` cho `int64`, khớp với đáp án mẫu (không thành `float`).
- **Không đổi số hàng:** phép tính theo từng phần tử không thêm hay bớt hàng.
- **Trả về DataFrame:** hàm trả về toàn bộ bảng (đủ 3 cột), không chỉ riêng cột `bonus`.
- **Về tác dụng phụ:** Cách 2 sửa trực tiếp bảng truyền vào (LeetCode cho phép). Nếu muốn không sửa đầu vào, dùng `assign` (Cách 3).

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` là số cột ban đầu.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp Python | `O(m)` (chậm, Python thuần) | `O(m)` |
| 2. `df["bonus"] = df["salary"] * 2` | `O(m)` (vector hóa) | `O(m)` |
| 3. `assign` | `O(m·c)` hoặc nhẹ hơn nhờ CoW | `O(m·c)` hoặc nhẹ hơn nhờ CoW |
| 4. `apply` / `map` | `O(m)` (mỗi phần tử một lời gọi hàm Python) | `O(m)` |
| 5. `insert` / `eval` | `O(m)` | `O(m)` |

Mọi cách phải chạm vào cả `m` giá trị nên không thể tốt hơn `O(m)`. Khác biệt thực tế là hằng số: vector hóa nhanh hơn `apply` và vòng lặp Python từ vài đến hàng chục lần trên bảng lớn.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Vì Cách 1, 2, 4, 5 **sửa tại chỗ**, mỗi lần gọi ta truyền một **bản sao** (`df.copy()`) để các cách không ảnh hưởng nhau.

In [21]:
impls = {
    "loop": bonus_loop,
    "vectorized": bonus_vectorized,
    "assign": bonus_assign,
    "apply": bonus_apply,
    "insert": bonus_insert,
    "eval": bonus_eval,
    "submit": createBonusColumn,
}

expected_example = pd.DataFrame({
    "name": ["Piper", "Grace", "Georgia", "Willow", "Finn", "Thomas"],
    "salary": [4548, 28150, 1103, 6593, 74576, 24433],
    "bonus": [9096, 56300, 2206, 13186, 149152, 48866],
})
for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(employees.copy()), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


In [22]:
def make(salaries, names=None):
    return pd.DataFrame({"name": names or [f"e{i}" for i in range(len(salaries))], "salary": salaries})

edge = {
    "một hàng": make([100]),
    "lương bằng 0": make([0, 5]),
    "lương lớn": make([10**9, 10**9]),
    "lương trùng nhau": make([7, 7, 7]),
    "nhiều hàng": make(list(range(1, 51))),
}
for name, df in edge.items():
    exp = df.assign(bonus=[2 * s for s in df["salary"]])
    for key, fn in impls.items():
        got = fn(df.copy())
        pd.testing.assert_frame_equal(got, exp)
        assert list(got.columns) == ["name", "salary", "bonus"], (name, key)
        assert str(got["bonus"].dtype) == "int64", (name, key)
    print("PASS", name, "->", exp.shape)

PASS một hàng -> (1, 3)
PASS lương bằng 0 -> (2, 3)
PASS lương lớn -> (2, 3)
PASS lương trùng nhau -> (3, 3)
PASS nhiều hàng -> (50, 3)


### 12.2. Tác dụng phụ: cách nào sửa bảng đầu vào?

Kiểm tra bằng thực nghiệm: gọi hàm với một bảng `src` rồi xem `src` có thêm cột `bonus` không.

In [23]:
mutating = {}
for key, fn in impls.items():
    src = employees.copy()
    fn(src)
    mutating[key] = "bonus" in src.columns
print(mutating)
assert mutating == {"loop": True, "vectorized": True, "assign": False, "apply": True,
                    "insert": True, "eval": False, "submit": True}
print("PASS: assign và eval không sửa đầu vào; các cách gán trực tiếp thì sửa")

# Phản ví dụ: bonus là số nguyên, không phải số thực
wrong = employees.assign(bonus=employees["salary"] * 2.0)
assert str(wrong["bonus"].dtype) == "float64" and str(createBonusColumn(employees.copy())["bonus"].dtype) == "int64"
print("float64 vs int64: ", wrong["bonus"].dtype, "vs", createBonusColumn(employees.copy())["bonus"].dtype)

{'loop': True, 'vectorized': True, 'assign': False, 'apply': True, 'insert': True, 'eval': False, 'submit': True}
PASS: assign và eval không sửa đầu vào; các cách gán trực tiếp thì sửa
float64 vs int64:  float64 vs int64


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `m ∈ [1, 200]` nhân viên với lương ngẫu nhiên `[0, 10⁶]`, kèm index lộn xộn (để kiểm tra việc "căn theo index"), rồi so mọi cách với đáp án tính bằng vòng lặp thuần.

In [24]:
import random
random.seed(2881)
np.random.seed(2881)
for t in range(200):
    m = random.randint(1, 200)
    df = pd.DataFrame({
        "name": [random.choice(["An", "Binh", "Chau", "Dat"]) for _ in range(m)],
        "salary": np.random.randint(0, 10**6 + 1, size=m),
    })
    if random.random() < 0.5:
        df.index = random.sample(range(10000), m)
    exp_bonus = [int(s) * 2 for s in df["salary"]]
    for key, fn in impls.items():
        got = fn(df.copy())
        assert got["bonus"].tolist() == exp_bonus, (t, key)
        assert got["salary"].tolist() == df["salary"].tolist() and got["name"].tolist() == df["name"].tolist()
        assert got.index.tolist() == df.index.tolist(), (t, key)
        assert list(got.columns) == ["name", "salary", "bonus"], (t, key)
print("200 bảng ngẫu nhiên: mọi cách cho cột bonus = 2 * salary, đúng hàng, đúng index")

200 bảng ngẫu nhiên: mọi cách cho cột bonus = 2 * salary, đúng hàng, đúng index


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `df.bonus = ...` — không tạo được cột

In [25]:
import warnings
demo = employees.copy()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    demo.bonus = demo["salary"] * 2                    # sai: chỉ tạo thuộc tính Python
print("cột:", demo.columns.tolist())                   # vẫn không có 'bonus'
print("số cảnh báo:", len(caught))
print("đúng:", createBonusColumn(employees.copy()).columns.tolist())

cột: ['name', 'salary']
số cảnh báo: 1
đúng: ['name', 'salary', 'bonus']


**Vì sao sai:** `df.tên = ...` **không** tạo cột (pandas chỉ gắn một thuộc tính Python và cảnh báo). Muốn tạo cột phải dùng ngoặc vuông với chuỗi: `df["bonus"] = ...`.

### Lỗi 2: Tính xong nhưng **không gán**

In [26]:
def sai_khong_gan(employees):
    employees["salary"] * 2                     # sai: tính rồi bỏ đi
    return employees

print("sai :", sai_khong_gan(employees.copy()).columns.tolist())
print("đúng:", createBonusColumn(employees.copy()).columns.tolist())

sai : ['name', 'salary']
đúng: ['name', 'salary', 'bonus']


**Vì sao sai:** biểu thức `employees["salary"] * 2` tạo ra một Series mới nhưng **không được lưu đi đâu**. Phải gán vào `employees["bonus"]`.

### Lỗi 3: Quên dấu nháy quanh tên cột

In [27]:
def sai_thieu_nhay(employees):
    employees[bonus] = employees["salary"] * 2          # sai: bonus là biến chưa định nghĩa
    return employees

try:
    sai_thieu_nhay(employees.copy())
except NameError as e:
    print("NameError:", e)

NameError: name 'bonus' is not defined


**Vì sao sai:** `bonus` không có nháy là **tên biến**, chưa được tạo. Tên cột là **chuỗi**: `"bonus"`.

### Lỗi 4: Nhân với `2.0` hoặc chia → dtype `float64`

In [28]:
def sai_float(employees):
    employees["bonus"] = employees["salary"] * 2.0
    return employees

out = sai_float(employees.copy())
print(out["bonus"].dtype, out["bonus"].tolist()[:3])
print(out.equals(expected_example))                  # False vì khác dtype

float64 [9096.0, 56300.0, 2206.0]
False


**Vì sao sai:** giá trị trông giống nhau (`9096.0` vs `9096`) nhưng `float64` khác `int64`. Dùng `* 2` (số nguyên).

### Lỗi 5: List có độ dài khác số hàng

In [29]:
try:
    bad = employees.copy()
    bad["bonus"] = [1, 2, 3]
except ValueError as e:
    print("ValueError:", str(e)[:70])

ValueError: Length of values (3) does not match length of index (6)


**Vì sao sai:** gán list làm cột cần **đúng `len(df)` phần tử**. Gán một Series thì pandas căn theo index (thiếu thì `NaN`).

### Lỗi 6: Trả về riêng cột `bonus` thay vì cả bảng

In [30]:
def sai_tra_cot(employees):
    employees["bonus"] = employees["salary"] * 2
    return employees["bonus"]                   # sai: trả Series

r = sai_tra_cot(employees.copy())
print(type(r).__name__, "-> đúng ra phải là DataFrame:", type(createBonusColumn(employees.copy())).__name__)

Series -> đúng ra phải là DataFrame: DataFrame


**Vì sao sai:** đề yêu cầu trả về **DataFrame** đủ ba cột `name`, `salary`, `bonus`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thêm / tạo cột mới từ cột khác" → `df["cột_mới"] = công_thức_trên_cột` (vector hóa).
- Công thức: `df["a"] * 2`, `df["a"] + df["b"]`, `df["a"] > 10` (ra cột bool). Không cần `for`.
- Tên cột luôn là **chuỗi có nháy** trong ngoặc vuông; **không** dùng `df.cột_mới = ...` để tạo cột.
- Không muốn sửa đầu vào → `df.assign(cột_mới=...)`.
- Kiểm dtype (`df.dtypes`): số nguyên nhân số nguyên vẫn nguyên; dùng `/` hoặc `2.0` sẽ thành số thực.
- Sau khi gán, in `df.head()` để chắc rằng cột đã có.

## 15. Biến thể của bài

1. **Cột điều kiện:** `df["high"] = df["salary"] > 10000` (kiểu bool); hoặc `np.where(df["salary"] > 10000, "cao", "thấp")`.
2. **Thưởng theo bậc:** 10% nếu lương < 5000, 20% nếu lớn hơn — dùng `np.where` hoặc `pd.cut`.
3. **Cột từ hai cột:** `df["total"] = df["salary"] + df["bonus"]`.
4. **Đổi tên / thứ tự cột:** `df.rename(columns=...)`, `df[["name", "bonus", "salary"]]`.
5. **Ghi đè cột có sẵn:** `df["salary"] = df["salary"] * 2`.
6. Liên quan LeetCode 2877 *Create a DataFrame from List* và 2882 *Drop Duplicate Rows*.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp Python | tính từng lương, gom list | `O(m)` (chậm) | `O(m)` | hiểu bản chất; tránh khi bảng lớn |
| 2. `df["bonus"] = df["salary"] * 2` | vector hóa + gán cột | `O(m)` | `O(m)` | **cách nộp bài** |
| 3. `assign` | trả về bảng mới | `O(m·c)` / nhẹ nhờ CoW | tương tự | không muốn sửa đầu vào, nối chuỗi lệnh |
| 4. `apply` / `map` | hàm cho từng phần tử | `O(m)` (chậm hơn) | `O(m)` | công thức phức tạp, không vector hóa được |
| 5. `insert` / `eval` | chọn vị trí cột / biểu thức chuỗi | `O(m)` | `O(m)` | cần đặt cột ở vị trí cụ thể |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(employees["salary"] * 2).tolist()[:2]`, `list(employees.assign(x=1).columns)`, `(employees["salary"] > 20000).sum()`, `type(employees["salary"] * 2).__name__`.

**Bài 2 (tính tay).** Với `salary = [500, 1200, 90]` hãy viết cột `bonus = 2 × salary` và cột `total = salary + bonus`.

**Bài 3 (điền TODO).** Hoàn thành `add_bonus_todo(df)` thêm `bonus` bằng `assign` (không sửa `df`).

In [31]:
def add_bonus_todo(df):
    # TODO 1: tạo Series gấp đôi lương: df["salary"] * 2
    # TODO 2: return df.assign(bonus=Series_vừa_tạo)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `create_bonus_loop(employees)` bằng vòng lặp (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `add_total(df)`: thêm cột `bonus = 2 * salary` và cột `total = salary + bonus`. (b) Viết `add_level(df)`: thêm cột `level` có giá trị `"high"` nếu `salary >= 20000`, ngược lại `"low"` (gợi ý: `np.where`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[9096, 56300]`; `['name', 'salary', 'x']`; số lương lớn hơn 20000 là `28150, 74576, 24433` nên `3`; `Series`. Bài 2: `bonus = [1000, 2400, 180]`.
- **Mức 2 (vừa):** Bài 2: `total = [1500, 3600, 270]`. Bài 3: hai dòng đã ghi trong comment. Bài 4: `bonus = []`, `for salary in ...: bonus.append(salary * 2)`, rồi gán.
- **Mức 3 (sâu):** Bài 5(a): `df.assign(bonus=df["salary"] * 2).assign(total=lambda x: x["salary"] + x["bonus"])` (dùng `lambda` để tham chiếu bảng vừa tạo). Bài 5(b): `np.where(df["salary"] >= 20000, "high", "low")`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert (employees["salary"] * 2).tolist()[:2] == [9096, 56300]
assert list(employees.assign(x=1).columns) == ["name", "salary", "x"]
assert (employees["salary"] > 20000).sum() == 3
assert type(employees["salary"] * 2).__name__ == "Series"

# Bài 2
b2 = pd.DataFrame({"salary": [500, 1200, 90]})
b2["bonus"] = b2["salary"] * 2
b2["total"] = b2["salary"] + b2["bonus"]
assert b2["bonus"].tolist() == [1000, 2400, 180] and b2["total"].tolist() == [1500, 3600, 270]

# Bài 3
def add_bonus_todo_ans(df):
    doubled = df["salary"] * 2
    return df.assign(bonus=doubled)

src = employees.copy()
r3 = add_bonus_todo_ans(src)
assert "bonus" not in src.columns
pd.testing.assert_frame_equal(r3, expected_example)

# Bài 4
def create_bonus_loop(employees):
    bonus = []
    for salary in employees["salary"]:
        bonus.append(salary * 2)
    employees["bonus"] = bonus
    return employees

pd.testing.assert_frame_equal(create_bonus_loop(employees.copy()), expected_example)

# Bài 5(a)
def add_total(df):
    out = df.assign(bonus=df["salary"] * 2)
    out["total"] = out["salary"] + out["bonus"]
    return out

r5a = add_total(employees)
assert r5a["total"].tolist() == [4548 * 3, 28150 * 3, 1103 * 3, 6593 * 3, 74576 * 3, 24433 * 3]
assert list(r5a.columns) == ["name", "salary", "bonus", "total"]

# Bài 5(b)
def add_level(df):
    return df.assign(level=np.where(df["salary"] >= 20000, "high", "low"))

r5b = add_level(employees)
assert r5b["level"].tolist() == ["low", "high", "low", "low", "high", "high"]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cột mới từ phép toán trên cột cũ)

```python
import pandas as pd

def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame:
    employees["bonus"] = employees["salary"] * 2
    return employees
```

**Checklist**

- [ ] Tên cột mới là chuỗi có nháy: `df["bonus"] = ...` (không phải `df.bonus = ...`)?
- [ ] Đã **gán** kết quả vào cột (không chỉ tính rồi bỏ)?
- [ ] `bonus` là `salary * 2` với **số nguyên** (dtype `int64`, không thành `float`)?
- [ ] Trả về **cả DataFrame** (3 cột), không chỉ riêng cột `bonus`?
- [ ] Biết cách này **sửa** bảng đầu vào; dùng `assign` nếu không muốn sửa?
- [ ] Skeleton đúng `def createBonusColumn(employees: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?